# CO Timing - All FRN4 Input Variations

Generates `co_timing_all_variations.csv` containing charge-off, payoff, and total-loss month for every combination of:

| Dimension | Values |
|---|---|
| model_version | franchise4.0, franchise4.1 |
| lob | AN, ENT, FLD, FRN, STG, STE, MCY |
| term | 36, 48, 60, 72 |
| tag | 0 through 15 |

**Total rows: 896**

In [1]:
import itertools
import pandas as pd

# ---------------------------------------------------------------------------
# Embedded lookup tables (from co_mapping_init)
# ---------------------------------------------------------------------------

# co_timing_by_group.csv - franchise 4.0
gdf_co_month = {
    0: {'mean_months_co': 16.6499940321891},
    1: {'mean_months_co': 18.7139882591884},
    2: {'mean_months_co': 20.104802001085},
    3: {'mean_months_co': 22.1608857567173},
    4: {'mean_months_co': 18.3819312150872},
    5: {'mean_months_co': 20.3924603761012},
    6: {'mean_months_co': 21.8375193850043},
    7: {'mean_months_co': 23.8670712511511},
    8: {'mean_months_co': 18.3486563223507},
    9: {'mean_months_co': 20.3618135888052},
    10: {'mean_months_co': 21.4460455785775},
    11: {'mean_months_co': 23.1082630254322},
    12: {'mean_months_co': 16.3533004043629},
    13: {'mean_months_co': 19.0436469978706},
    14: {'mean_months_co': 20.033330721303},
    15: {'mean_months_co': 22.5584333108439},
}

# co_timing_by_group_frn41.csv - franchise 4.1
gdf_co_month_frn_41 = {
    0: {'mean_months_co': 17.804597},
    1: {'mean_months_co': 19.992664},
    2: {'mean_months_co': 21.511454},
    3: {'mean_months_co': 23.281872},
    4: {'mean_months_co': 21.083496},
    5: {'mean_months_co': 24.075087},
    6: {'mean_months_co': 24.68797},
    7: {'mean_months_co': 26.032507},
    8: {'mean_months_co': 24.04203},
    9: {'mean_months_co': 25.23287},
    10: {'mean_months_co': 26.84651},
    11: {'mean_months_co': 29.005345},
    12: {'mean_months_co': 26.140272},
    13: {'mean_months_co': 27.610666},
    14: {'mean_months_co': 31.982823},
    15: {'mean_months_co': 32.695062},
}

po_by_tag = {
    0: 36.0, 1: 38.0, 2: 40.0, 3: 42.0, 4: 44.0, 5: 46.0, 6: 48.0, 7: 50.0,
    8: 52.0, 9: 54.0, 10: 56.0, 11: 58.0, 12: 60.0, 13: 62.0, 14: 64.0, 15: 66.0,
}

gdf_po_month_frn_41 = {k: {'mean_months_po': v} for k, v in po_by_tag.items()}

ADJUSTMENTS = {
    "AN": 0.9452350907111133,
    "ENT": 1.0994796517684224,
    "FLD": 1.027725745972301,
    "FRN": 0.9047309801560975,
    "STG": 0.9230164639536621,
    "STE": 1.1,
    "MCY": 0.88,
}

In [2]:
def get_scenario_timing_frn_4(term, tag, lob, model_version):
    """Compute CO / PO / total-loss month for a single FRN4 combination."""
    co_scaling = 24.47 / 20.23
    po_scaling = 43.17 / 44.29

    if model_version == 'franchise4.1':
        co_scaling *= 20.56 / 26.94
        po_scaling *= 43.25 / 41.76
        base_co = gdf_co_month_frn_41.get(tag, gdf_co_month_frn_41[0])['mean_months_co']
        base_po = gdf_po_month_frn_41.get(tag, gdf_po_month_frn_41[0])['mean_months_po']
    else:
        base_co = gdf_co_month.get(tag, gdf_co_month[0])['mean_months_co']
        base_po = po_by_tag.get(tag, 36.0)

    charge_off_month = base_co * co_scaling
    pay_off_month = base_po * po_scaling

    charge_off_month *= ADJUSTMENTS.get(lob, ADJUSTMENTS['FRN'])
    charge_off_month *= (0.5 * term / 72 + 0.5)

    pay_off_month *= ADJUSTMENTS.get(lob, ADJUSTMENTS['FRN'])
    pay_off_month *= (0.5 * term / 72 + 0.5)

    total_loss_month = pay_off_month / 2

    return {
        'charge_off_month': charge_off_month,
        'pay_off_month': pay_off_month,
        'total_loss_month': total_loss_month,
    }

In [3]:
model_versions = ['franchise4.0', 'franchise4.1']
lobs = ['AN', 'ENT', 'FLD', 'FRN', 'STG', 'STE', 'MCY']
terms = [36, 48, 60, 72]
tags = list(range(16))

rows = []
for mv, lob, term, tag in itertools.product(model_versions, lobs, terms, tags):
    timing = get_scenario_timing_frn_4(term, tag, lob, mv)
    rows.append({
        'model_version': mv,
        'lob': lob,
        'term': term,
        'tag': tag,
        'charge_off_month': timing['charge_off_month'],
        'pay_off_month': timing['pay_off_month'],
        'total_loss_month': timing['total_loss_month'],
    })

df = pd.DataFrame(rows)
print(f"{len(df)} rows generated")
df.head(10)

896 rows generated


,model_version,lob,term,tag,charge_off_month,pay_off_month,total_loss_month
0,franchise4.0,AN,36,0,14.277536,24.875967,12.437983
1,franchise4.0,AN,36,1,16.047432,26.257965,13.128982
2,franchise4.0,AN,36,2,17.240068,27.639963,13.819982
3,franchise4.0,AN,36,3,19.003181,29.021961,14.510981
4,franchise4.0,AN,36,4,15.762690,30.403959,15.201980
5,franchise4.0,AN,36,5,17.486738,31.785958,15.892979
6,franchise4.0,AN,36,6,18.725891,33.167956,16.583978
7,franchise4.0,AN,36,7,20.466252,34.549954,17.274977
8,franchise4.0,AN,36,8,15.734156,35.931952,17.965976
9,franchise4.0,AN,36,9,17.460458,37.313950,18.656975


In [4]:
output_path = 'co_timing_all_variations.csv'
df.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

Saved to co_timing_all_variations.csv
